# 🔍 멀티사이트 크롤링 파이프라인
**대상 사이트**: 네이버 카페 홈, 네이버 지식인, 블라인드, 오늘의집  
**구조**: 2단계 크롤링 (1단계: URL 수집 → 2단계: 본문 파싱)  
**로그인**: 불필요 (공개 검색결과만 수집)

In [ ]:
# ============================================================
# 셀 1: 라이브러리 import
# ============================================================
import time
import random
import re
import os
import pandas as pd
from datetime import datetime
from urllib.parse import quote, urljoin

import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

print('✅ 라이브러리 로드 완료')

In [ ]:
# ============================================================
# 셀 2: 사용자 설정 변수 (여기만 수정하면 됩니다!)
# ============================================================

KEYWORDS = [
    "현관 깜빡",
    "현관 확인",
    "현관 귀가",
    "현관 귀찮"# 현관 관련 커뮤니티 글에서 자주 등장
]

START_DATE = "2023-01-01"  # ✏️ 이 날짜 이후 데이터만 수집

OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 크롤링할 사이트 선택 (True/False로 켜고 끄기)
SITES = {
    "naver_cafe": True,
    "naver_kin":  True,
    "blind":      False,  # 봇 감지로 제외
    "ohou":       True,
}

START_DT = datetime.strptime(START_DATE, "%Y-%m-%d")
print(f'✅ 키워드 ({len(KEYWORDS)}개): {KEYWORDS}')
print(f'✅ 수집 기간: {START_DATE} 이후')

In [ ]:
# ============================================================
# 셀 3: 공통 유틸 함수
# ============================================================

def get_driver(headless=False):  # False = 브라우저 창 표시
    """Chrome WebDriver 생성"""
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/120.0.0.0 Safari/537.36")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver


def random_sleep(min_sec=1.5, max_sec=3.5):z
    """랜덤 딜레이"""
    time.sleep(random.uniform(min_sec, max_sec))


def parse_date(date_str):
    """다양한 날짜 포맷을 datetime으로 파싱"""
    if not date_str:
        return None
    date_str = str(date_str).strip()

    # "N일", "N시간 전", "N분 전", "방금" → 오늘
    if any(x in date_str for x in ["일", "시간", "분", "전", "ago", "방금"]):
        return datetime.now()

    formats = [
        "%Y.%m.%d", "%Y-%m-%d", "%Y/%m/%d",
        "%Y.%m.%d %H:%M", "%Y-%m-%d %H:%M:%S",
        "%m.%d",  # 올해 게시글 (연도 없음) ex) 01.30
    ]
    for fmt in formats:
        try:
            dt = datetime.strptime(date_str, fmt)
            if dt.year == 1900:  # 연도 없는 경우 현재 연도 적용
                dt = dt.replace(year=datetime.now().year)
            return dt
        except ValueError:
            continue
    return None


def is_after_start(date_str, start_dt=START_DT):
    """날짜가 시작일 이후인지 확인"""
    dt = parse_date(date_str)
    if dt is None:
        return True  # 날짜 파싱 실패 시 일단 포함
    return dt >= start_dt


def save_csv(records, site_name, keyword):
    """결과를 CSV로 저장 → 사이트_키워드_crawling.csv"""
    if not records:
        print(f'  ⚠️ [{site_name}] "{keyword}" 수집 결과 없음')
        return
    df = pd.DataFrame(records)
    safe_keyword = keyword.replace(" ", "_").replace("/", "_")
    filename = f"{OUTPUT_DIR}/{site_name}_{safe_keyword}_crawling.csv"
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f'  ✅ [{site_name}] "{keyword}" → {len(records)}건 저장: {filename}')


def get_soup(url, headers=None):
    """requests로 HTML 가져오기"""
    if headers is None:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0.0.0 Safari/537.36",
            "Accept-Language": "ko-KR,ko;q=0.9",
        }
    try:
        res = requests.get(url, headers=headers, timeout=10)
        res.raise_for_status()
        return BeautifulSoup(res.text, "html.parser")
    except Exception as e:
        print(f'    ❌ requests 실패: {e}')
        return None


print('✅ 유틸 함수 로드 완료')

In [ ]:
# ============================================================
# 셀 4-1: 네이버 카페 크롤러 (통합검색 날짜범위 방식)
# ============================================================
# URL 형식: date_option=8, date_from=YYYY.MM.DD, date_to=YYYY.MM.DD
# 연도별 쪼개기, 스크롤 20회/연도

def crawl_naver_cafe(keyword, start_dt=START_DT):
    from urllib.parse import quote as urlquote
    import datetime

    records   = []
    all_urls  = []
    seen_urls = set()
    driver    = get_driver(headless=True)

    try:
        end_year   = datetime.datetime.now().year
        start_year = max(start_dt.year, 2015)

        for year in range(end_year, start_year - 1, -1):
            df_dt = datetime.date(year, 1, 1)
            dt_dt = datetime.date(year, 12, 31)
            if year == start_dt.year:
                df_dt = start_dt.date() if hasattr(start_dt, "date") else start_dt
            if year == end_year:
                dt_dt = datetime.date.today()

            date_from_dot = df_dt.strftime("%Y.%m.%d")
            date_to_dot   = dt_dt.strftime("%Y.%m.%d")
            date_from_num = df_dt.strftime("%Y%m%d")
            date_to_num   = dt_dt.strftime("%Y%m%d")

            search_url = (
                f"https://search.naver.com/search.naver"
                f"?ssc=tab.cafe.all&cafe_where=&query={urlquote(keyword)}"
                f"&ie=utf8&st=rel&date_option=8"
                f"&date_from={date_from_dot}&date_to={date_to_dot}"
                f"&srchby=text&cafe_url=&without_cafe_url="
                f"&sm=tab_opt&nso=so%3Ar%2Cp%3Afrom{date_from_num}to{date_to_num}"
                f"&nso_open=1&prdtype=0"
            )

            print(f'  📅 [{year}년] 카페: {date_from_dot} ~ {date_to_dot}')
            driver.get(search_url)
            random_sleep(1, 1.5)

            try:
                WebDriverWait(driver, 8).until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "ul.lst_total, div.total_area, a.title_link")
                    )
                )
            except TimeoutException:
                print(f'    결과 없음, 스킵')
                continue

            year_urls  = []
            no_new_cnt = 0

            for scroll_i in range(100):  # 20스크롤/연도
                soup = BeautifulSoup(driver.page_source, "html.parser")
                prev_len = len(year_urls)

                for a in soup.select("a.title_link, a.cafe_link"):
                    href = a.get("href", "")
                    if "cafe.naver.com" in href and href not in seen_urls:
                        seen_urls.add(href)
                        year_urls.append(href)

                if len(year_urls) == prev_len:
                    no_new_cnt += 1
                    if no_new_cnt >= 3:
                        break
                else:
                    no_new_cnt = 0

                prev_h = driver.execute_script("return document.body.scrollHeight")
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                random_sleep(1.5, 2.5)
                new_h = driver.execute_script("return document.body.scrollHeight")
                if new_h == prev_h:
                    break

            all_urls.extend(year_urls)
            print(f'    → {year}년 수집: {len(year_urls)}개 | 누적: {len(all_urls)}개')
            random_sleep(1, 2)

        print(f'  📋 URL 수집 완료: {len(all_urls)}개')

        # ── 본문 파싱 ────────────────────────────────────────────
        for i, url in enumerate(all_urls):
            try:
                driver.get(url)
                random_sleep(1.5, 2)

                try:
                    WebDriverWait(driver, 8).until(
                        EC.frame_to_be_available_and_switch_to_it((By.ID, "cafe_main"))
                    )
                except TimeoutException:
                    pass

                soup = BeautifulSoup(driver.page_source, "html.parser")

                title = ""
                for sel in ["h3.title", ".ArticleTitle", "#articleTitle", ".title_text", "h1"]:
                    el = soup.select_one(sel)
                    if el:
                        title = el.get_text(strip=True)
                        break

                content = ""
                for sel in ["div.content.CafeViewer", ".se-main-container", "#postViewArea", ".article_body"]:
                    el = soup.select_one(sel)
                    if el:
                        content = el.get_text(separator=" ", strip=True)[:3000]
                        break

                date_str = ""
                for sel in [".article_info .date", ".date_info", "span.date"]:
                    el = soup.select_one(sel)
                    if el:
                        date_str = el.get_text(strip=True)
                        break

                try:
                    driver.switch_to.default_content()
                except:
                    pass

                records.append({"url": url, "date": date_str, "title": title, "content": content})

                if (i + 1) % 10 == 0:
                    print(f'    본문 파싱: {i+1}/{len(all_urls)}')

            except Exception as e:
                print(f'    ❌ 파싱 오류 ({url[:50]}): {e}')
                try: driver.switch_to.default_content()
                except: pass

    finally:
        driver.quit()

    return records


print('✅ 네이버 카페 크롤러 로드 완료')

In [ ]:
# ============================================================
# 셀 4-2: 네이버 지식인 크롤러 (통합검색 날짜범위 방식) - 수정본
# ============================================================
def crawl_naver_kin(keyword, start_dt=START_DT):
    from urllib.parse import quote as urlquote
    import datetime
    from bs4 import BeautifulSoup
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException

    records   = []
    all_urls  = []
    seen_urls = set()
    driver    = get_driver(headless=True) # 기존에 정의하신 드라이버 호출 함수 가정

    try:
        end_year   = datetime.datetime.now().year
        # start_dt의 연도와 2026 중 큰 값부터 시작하게 되어있는데, 
        # 필요에 따라 로직 확인 (예: start_dt가 과거일 경우 문제없는지)
        start_year = max(start_dt.year, 2015) 

        for year in range(end_year, start_year - 1, -1):
            df_dt = datetime.date(year, 1, 1)
            dt_dt = datetime.date(year, 12, 31)
            if year == start_dt.year:
                df_dt = start_dt.date() if hasattr(start_dt, "date") else start_dt
            if year == end_year:
                dt_dt = datetime.date.today()

            date_from_num = df_dt.strftime("%Y%m%d")
            date_to_num   = dt_dt.strftime("%Y%m%d")

            search_url = (
                f"https://search.naver.com/search.naver"
                f"?ssc=tab.kin.kqna&where=kin&sm=tab_jum"
                f"&query={urlquote(keyword)}"
                f"&nso=so%3Ar%2Cp%3Afrom{date_from_num}to{date_to_num}"
            )

            print(f'  📅 [{year}년] 지식IN: {df_dt} ~ {dt_dt}')
            driver.get(search_url)
            random_sleep(1, 1.5)

            # [수정 1] 페이지 로딩 대기 조건: 동적 클래스가 아닌 확실한 href 속성으로 대기
            try:
                WebDriverWait(driver, 8).until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "a[href*='kin.naver.com/qna/detail.naver']")
                    )
                )
            except TimeoutException:
                print(f'    결과 없음, 스킵')
                continue

            year_urls  = []
            no_new_cnt = 0

            # [수정 2] 스크롤 루프 버그 수정: range(1) -> range(20)
            for scroll_i in range(100):  
                soup = BeautifulSoup(driver.page_source, "html.parser")
                prev_len = len(year_urls)

                # [수정 3] URL 수집 규칙 강력화: 복잡한 부모 태그 무시하고 타겟 링크만 쏙쏙 추출
                for a in soup.select("a[href*='kin.naver.com/qna/detail.naver']"):
                    href = a.get("href", "")
                    
                    # 'kin.naver.com'이 포함된 진짜 URL인지 확인하고 중복 제거
                    if href and href not in seen_urls:
                        seen_urls.add(href)
                        year_urls.append(href)

                if len(year_urls) == prev_len:
                    no_new_cnt += 1
                    if no_new_cnt >= 3:
                        break # 새로운 URL이 3번 연속으로 추가되지 않으면 스크롤 종료
                else:
                    no_new_cnt = 0

                prev_h = driver.execute_script("return document.body.scrollHeight")
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                random_sleep(1.5, 2)
                new_h = driver.execute_script("return document.body.scrollHeight")
                if new_h == prev_h:
                    break # 더 이상 페이지 길이가 늘어나지 않으면 (마지막 도달) 종료

            all_urls.extend(year_urls)
            print(f'    → {year}년 수집: {len(year_urls)}개 | 누적: {len(all_urls)}개')
            random_sleep(1, 2)

        print(f'  📋 URL 수집 완료: {len(all_urls)}개')

        # ── 본문 파싱 ────────────────────────────────────────────
        for i, url in enumerate(all_urls):
            try:
                driver.get(url)
                random_sleep(1.5, 2)
                soup = BeautifulSoup(driver.page_source, "html.parser")

                title_el = (
                    soup.select_one("#content div.endTitleSection") or
                    soup.select_one(".endTitleSection") or
                    soup.select_one(".c-heading__title") or
                    soup.select_one("h3.title") or
                    soup.select_one("#subject") or
                    soup.select_one(".title") # 최신 상세 페이지 대비용 클래스 추가
                )
                title = title_el.get_text(strip=True) if title_el else ""

                content = ""
                for sel in ["#content div.questionDetail", ".questionDetail", ".c-heading__content"]:
                    el = soup.select_one(sel)
                    if el:
                        content = el.get_text(separator=" ", strip=True)[:3000]
                        break

                date_str = ""
                for sel in [".date", "span.date", ".answer_info .date", ".question_info .date", ".c-userinfo__info"]:
                    el = soup.select_one(sel)
                    if el:
                        date_str = el.get_text(strip=True)
                        break

                records.append({"url": url, "date": date_str, "title": title, "content": content})

                if (i + 1) % 10 == 0:
                    print(f'    본문 파싱: {i+1}/{len(all_urls)}')

            except Exception as e:
                print(f'    ❌ 파싱 오류: {e}')

    finally:
        driver.quit()

    return records

print('✅ 네이버 지식인 크롤러 로드 완료')

In [ ]:
# ============================================================
# 셀 4-3: 블라인드 크롤러 (제외됨 - 봇 감지로 수집 불가)
# ============================================================
print("⏭️ 블라인드 크롤러 비활성화 (봇 감지)")

In [ ]:
# ============================================================
# 셀 4-4: 오늘의집 크롤러 (스크롤 끝까지 전량 수집)
# ============================================================

def crawl_ohou(keyword, start_dt=START_DT):
    """
    오늘의집 커뮤니티 무한스크롤 - 스크롤 제한 없이 끝까지 수집
    URL: https://ohou.se/search/community?query=키워드&search_affect_type=Typing
    종료 조건: 스크롤 높이 변화 없을 때만 (= 진짜 끝)
    수집 후 날짜 필터링
    """
    records = []
    driver  = get_driver(headless=True)

    try:
        search_url = (f"https://ohou.se/search/community"
                      f"?query={quote(keyword)}&search_affect_type=Typing")
        print(f'  🔍 오늘의집 검색: "{keyword}"')
        driver.get(search_url)
        random_sleep(3, 4)

        # ── 1단계: 스크롤 끝까지 URL 수집 ──────────────────────
        urls      = []
        seen_urls = set()
        scroll    = 0

        while True:
            soup = BeautifulSoup(driver.page_source, "html.parser")
            new_count = 0

            for a_tag in soup.select("article > a"):
                href = a_tag.get("href", "")
                if not href:
                    continue
                if href.startswith("/"):
                    href = "https://ohou.se" + href
                if href in seen_urls:
                    continue

                date_el = a_tag.select_one(
                    "div[class*='eeb3cje1'] > div > span:nth-child(3), "
                    "div[class*='css-10h7hkg'] > div > span:nth-child(3)"
                )
                if not date_el:
                    spans   = a_tag.select("span[class*='css-']")
                    date_el = spans[-1] if spans else None
                date_str = date_el.get_text(strip=True) if date_el else ""

                seen_urls.add(href)
                urls.append({"url": href, "date": date_str})
                new_count += 1

            scroll += 1
            print(f'    스크롤 {scroll}: 신규 {new_count}개, 누적 {len(urls)}개')

            prev_h = driver.execute_script("return document.body.scrollHeight")
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            random_sleep(2, 3)
            new_h = driver.execute_script("return document.body.scrollHeight")
            if new_h == prev_h:
                print(f'    ✅ 스크롤 끝, 전량 수집 완료')
                break

        before = len(urls)
        urls   = [u for u in urls if is_after_start(u["date"], start_dt)]
        print(f'  📋 URL 수집: {before}개 → 날짜 필터 후 {len(urls)}개')

        # ── 2단계: 본문 파싱 ────────────────────────────────────
        for i, item in enumerate(urls):
            try:
                driver.get(item["url"])
                random_sleep(2, 3)

                try:
                    WebDriverWait(driver, 8).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "h1, article"))
                    )
                except TimeoutException:
                    pass

                soup     = BeautifulSoup(driver.page_source, "html.parser")
                title_el = soup.select_one("h1, .title, [class*='title']")
                title    = title_el.get_text(strip=True) if title_el else ""

                content   = ""
                container = soup.select_one(
                    "#__next div[class*='e1r9vabk0'], div[class*='e1r9vabk0']"
                )
                if container:
                    lines = [
                        d.get_text(separator=" ", strip=True)
                        for d in container.find_all("div", class_=lambda c: c and "css-" in c)
                        if d.get_text(strip=True)
                    ]
                    content = " ".join(lines)[:3000]

                records.append({
                    "url":     item["url"],
                    "date":    item["date"],
                    "title":   title,
                    "content": content,
                })

                if (i + 1) % 10 == 0:
                    print(f'    본문 파싱: {i+1}/{len(urls)}')

            except Exception as e:
                print(f'    ❌ 파싱 오류: {e}')
                continue

    finally:
        driver.quit()

    return records


print('✅ 오늘의집 크롤러 로드 완료 (스크롤 제한 없음)')

In [ ]:
# ============================================================
# 셀 5: 전체 실행 루프
# ============================================================

CRAWLER_MAP = {
    "naver_cafe": (crawl_naver_cafe, "네이버카페"),
    "naver_kin":  (crawl_naver_kin,  "지식인"),
    # "blind": 제외 (봇 감지로 수집 불가)
    "ohou":       (crawl_ohou,       "오늘의집"),
}

summary = []  # 결과 요약

for site_key, (crawler_fn, site_name) in CRAWLER_MAP.items():
    if not SITES.get(site_key, False):
        print(f'⏭️ {site_name} 스킵')
        continue

    for keyword in KEYWORDS:
        print(f'\n🚀 [{site_name}] "{keyword}" 크롤링 시작...')
        try:
            records = crawler_fn(keyword)
            save_csv(records, site_name, keyword)
            summary.append({
                "site": site_name,
                "keyword": keyword,
                "count": len(records),
                "status": "✅ 성공"
            })
        except Exception as e:
            print(f'  ❌ [{site_name}] "{keyword}" 전체 오류: {e}')
            summary.append({
                "site": site_name,
                "keyword": keyword,
                "count": 0,
                "status": f"❌ 실패: {str(e)[:50]}"
            })

        random_sleep(2, 4)  # 사이트 간 딜레이

print('\n🎉 전체 크롤링 완료!')

In [ ]:
# ============================================================
# 셀 6: 결과 요약
# ============================================================

summary_df = pd.DataFrame(summary)
print('\n📊 수집 결과 요약')
print('=' * 50)
print(summary_df.to_string(index=False))
print('=' * 50)
print(f'총 수집: {summary_df["count"].sum()}건')

# output 폴더 파일 목록
import glob
files = glob.glob(f"{OUTPUT_DIR}/*.csv")
print(f'\n📁 저장된 CSV 파일 ({len(files)}개):')
for f in sorted(files):
    size = os.path.getsize(f)
    df_tmp = pd.read_csv(f)
    print(f'  {os.path.basename(f)} ({len(df_tmp)}행, {size/1024:.1f}KB)')